# Response of atmosphere to sea surface temperature bump using jax.jvp


In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp

import jcm
from jcm.physics.speedy.speedy_coords import get_speedy_coords
import jax_datetime as jdt

from jem.base.coupler import Coupler
from jem.components import JCMComponent, SlabOceanModel
from jem.components.slab import SlabGrid

use_ipython = 'get_ipython' in globals()

## Configurations

In [ ]:
start_datetime = jdt.to_datetime("2000-01-01")
coupling_timestep = jdt.to_timedelta(1, "day")
simulation_name = "01-03_aquaplanet_response_to_SST_perturbation_using_gradient"
output_dir = (Path("output") / simulation_name).resolve()
output_dir.mkdir(exist_ok=True, parents=True)
output_figure = output_dir / "sensitivity.png"

## Creating Flux and Scalar Exchange between Components

An *exchanger* is the only place where components exchange information. It is
differentiated along with everything else, so it has to be pure: it builds new
carries with `.replace(...)` rather than writing into the ones it is handed.

In [ ]:
def exchange(components, time):
    del time  # this exchange does not depend on the date

    atm = components["atm"]
    ocn = components["ocn"]

    ocn = dict(ocn, forcing=ocn["forcing"].replace(
        total_heat_flux=atm["derived"].total_heat_flux,
    ))
    atm = dict(atm, forcing=atm["forcing"].replace(
        sea_surface_temperature=ocn["state"].sea_surface_temperature,
    ))

    return dict(components, atm=atm, ocn=ocn)

## Create Coupled Model

In [ ]:
atm_model = jcm.model.Model(
    coords=get_speedy_coords(),
    start_date=start_datetime,
)

# Aquaplanet: no fractional mask, so every cell of the slab grid is ocean.
aquaplanet_grid = SlabGrid.from_coords(atm_model.coords.horizontal)

model = Coupler(
    dict(
        atm=JCMComponent(atm_model),
        ocn=SlabOceanModel(aquaplanet_grid),
    ),
    dict(exchange=exchange),
    coupling_timestep=coupling_timestep,
    start_date=start_datetime,
)

print(repr(model))

## Taking Gradient of the Coupled Model

Mathematically speaking, we are trying to assess
$$
x_{\mathrm{final}} = F_{t_0, T}(x_0)
$$
where $x_0$ and $x_{\mathrm{final}}$ the initial and final states of the system, $t_0$ the initial time, $T$ the length of simulation time, and $F_{t_0, T}$ the trajectory function that maps the input state from time $t = t_0$ to $t = t_0 +  T$.

The following code attempts to compute the sensitivity of the trajectory function to a pulse function $g(x)$. That is, let initial condition $x_0$ be perturbed as
$$
    x'_0(x; \epsilon) = x_0(x) + \epsilon g(x)
$$
the response of the final state to $\delta(x)$ will then be
$$
x'_{\mathrm{final}} = x_{\mathrm{final}} + \epsilon \, \delta x_{\mathrm{final}}
$$
through which the sensitivity of final statet to $g$, noted as $s_{x_{\mathrm{final}}}$, is formally defined as
$$
s_{x_{\mathrm{final}}} = \frac{\partial x'_{\mathrm{final}} }{\partial \epsilon}
$$

The entire computation is achieved using `jax.jvp`. 

In [ ]:
simulation_interval = jdt.to_timedelta(5, "day")
initial_coupled_carry = model.initialize()
trajectory_function = model.generate_trajectory_function(
    int(simulation_interval / coupling_timestep)
)


@jax.jit
def forecast(sst):
    """Run the coupled model from an ocean initialized with `sst`.

    The initial carry is rebuilt here rather than mutated: `sst` is the
    argument being differentiated, so it has to reach the trajectory function
    through the carry the function is called with.
    """
    ocn = initial_coupled_carry.components["ocn"]
    ocn = dict(ocn, state=ocn["state"].replace(sea_surface_temperature=sst))
    carry = initial_coupled_carry.replace(
        components=dict(initial_coupled_carry.components, ocn=ocn),
    )

    final_carry, _ = trajectory_function(carry)

    return (
        final_carry.components["ocn"]["state"].sea_surface_temperature,
        final_carry.components["atm"]["derived"].physics["_surface_flux"].v0,
    )


sst_initial = (
    initial_coupled_carry.components["ocn"]["state"].sea_surface_temperature
)

# Put a point SST perturbation in the middle of domain
shape2D = sst_initial.shape
tangent_sst_initial = jnp.zeros_like(sst_initial).at[shape2D[0]//2, shape2D[1]//2].set(1.0)

# Use jax.jvp to obtain the sensitivity of surface meridional wind and SST
(sst_final, v_final), (tangent_sst_final, tangent_v_final) = jax.jvp(forecast, (sst_initial,), (tangent_sst_initial,))

## Visualization

In [ ]:
import matplotlib as mplt
if not use_ipython:
    mplt.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
lat = atm_model.coords.horizontal.latitudes * 180/jnp.pi
lon = atm_model.coords.horizontal.longitudes * 180/jnp.pi

fig, ax = plt.subplots(3, 2, figsize=(12, 16))

sst_levels = jnp.linspace(-2, 35, 11)
tangent_sst_levels = jnp.linspace(-1, 1, 11) * 0.5
v_levels = jnp.linspace(-1, 1, 11) * 5
tangent_v_levels = jnp.linspace(-1, 1, 11) * 0.2

ax[0, 0].contourf(lon, lat, (sst_initial-273.15).transpose(), levels=sst_levels)
ax[0, 1].contourf(lon, lat, tangent_sst_initial.transpose(), levels=tangent_sst_levels, cmap="bwr")
ax[1, 0].contourf(lon, lat, (sst_final-273.15).transpose(), levels=sst_levels)
ax[1, 1].contourf(lon, lat, tangent_sst_final.transpose(), levels=tangent_sst_levels, cmap="bwr")
ax[2, 0].contourf(lon, lat, v_final.transpose(), levels=v_levels, cmap="bwr")
ax[2, 1].contourf(lon, lat, tangent_v_final.transpose(), levels=tangent_v_levels, cmap="bwr")

ax[0, 0].set_title("(a) $\\mathrm{SST}_\\mathrm{init}$")
ax[0, 1].set_title("(b) $\\partial \\mathrm{SST}_\\mathrm{init}$")
ax[1, 0].set_title("(c) $\\mathrm{SST}_\\mathrm{final}$")
ax[1, 1].set_title("(d) $\\partial \\mathrm{SST}_\\mathrm{final}$")
ax[2, 0].set_title("(e) $\\mathrm{v}_\\mathrm{final}$")
ax[2, 1].set_title("(f) $\\partial \\mathrm{v}_\\mathrm{final}$")

fig.suptitle(f"Response time: {simulation_interval / jdt.to_timedelta(1, 'day'):.1f} days")
for _ax in ax.flatten():
    _ax.set_xlabel("longitude [deg]")
    _ax.set_ylabel("latitude [deg]")
    
print(f"Saving result figures into: {output_figure}")
plt.savefig(output_figure, dpi=200)

if use_ipython:
    plt.show()